In [2]:
%pip install -q -U torch torchvision torchaudio torchmetrics tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 22.6 MB/s eta 0:00:0000:01


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchmetrics import Accuracy
from tqdm.auto import tqdm

In [4]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.01

print(f"Device: {device}")

Device: cpu


In [5]:
train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 13.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 338kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.17MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.22MB/s]


In [6]:
# Net? A mozhet Da?
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5, stride=1)
        self.conv2 = nn.Conv2d(10, 10, kernel_size=5, stride=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(4 * 4 * 10, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [7]:
def train_model(model, dataloader, criterion, optimizer, epochs, device):
    train_accuracy = Accuracy(task="multiclass", num_classes=10).to(device)

    for epoch in tqdm(range(1, epochs + 1), desc="Epochs"):
        model.train()
        train_accuracy.reset()

        running_loss = 0.0
        running_acc = 0.0

        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            batch_acc = train_accuracy(logits, labels)
            running_loss += loss.item() * images.size(0)
            running_acc += batch_acc.item() * images.size(0)

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_acc / len(dataloader.dataset)
        print(f"Epoch {epoch:02d} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}")

In [8]:
@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()
    test_accuracy = Accuracy(task="multiclass", num_classes=10).to(device)

    running_acc = 0.0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        batch_acc = test_accuracy(logits, labels)
        running_acc += batch_acc.item() * images.size(0)

    final_acc = running_acc / len(dataloader.dataset)
    print(f"Test accuracy: {final_acc:.4f}")

In [9]:
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)

train_model(model, train_loader, criterion, optimizer, EPOCHS, device)
evaluate_model(model, test_loader, device)

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | Loss: 1.6490 | Accuracy: 0.5164
Epoch 02 | Loss: 0.3270 | Accuracy: 0.9028
Epoch 03 | Loss: 0.2110 | Accuracy: 0.9368
Epoch 04 | Loss: 0.1596 | Accuracy: 0.9516
Epoch 05 | Loss: 0.1317 | Accuracy: 0.9602
Epoch 06 | Loss: 0.1132 | Accuracy: 0.9656
Epoch 07 | Loss: 0.1013 | Accuracy: 0.9689
Epoch 08 | Loss: 0.0922 | Accuracy: 0.9722
Epoch 09 | Loss: 0.0844 | Accuracy: 0.9742
Epoch 10 | Loss: 0.0781 | Accuracy: 0.9766
Test accuracy: 0.9791
